# Learning Activity 3: Analytical Dataset Construction (Actividad 3: Construcción de Conjuntos Analíticos)
**Business Case:** North Quay Gifts | **Dataset:** Online Retail II
**CRISP-DM Phase:** Phase 3 — Data Preparation

**Team Members:**
- Russel Emmanuel Ku Aguilar (ID: 2309131)
- Damian Nicolas Sanchez Novelo (ID: 2109134)
- Jonathan Abisai Velasco Martín (ID: 2309060)
- Ángel Rivaldo Canché Chuc (ID: 2309030)
- Bianca Alexandra Acosta Castellanos (ID: 2309001)

**Academic Advisor / Professor:** Didier Omar Gamboa Angulo (`didier.gamboa@upy.edu.mx`)
**Institution:** Faculty of Data Engineering, Universidad Politécnica de Yucatán (UPY)

---
### Objective
Turn the transactional version selected in Activity 2 (**Route B**: purchases plus credits, each on its own date) into two
customer-level tables:

1. `clientes_rfm.csv` — RFM over the full period (reference date 2011-12-10 00:00) for later segmentation.
2. `clientes_clasificacion.csv` — 180-day history before the 2011-07-01 cutoff and a 90-day repurchase label.

No cleaning is repeated: the input is `transacciones_procesadas.csv` exactly as exported in Activity 2. No logarithms,
scaling, clustering, train/test split or models are applied here.

**Notebook sections (referenced in the report):** Section 0 – Source control · Section 1 – RFM dataset ·
Section 2 – Repurchase classification dataset · Section 3 – Dataset comparison and export.

In [1]:
import os, json
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)
os.makedirs('outputs', exist_ok=True)
R = {}   # results that feed the report
print('Analysis environment initialized | pandas', pd.__version__)

Analysis environment initialized | pandas 3.0.5


---
## Section 0 – Source control (Control de origen)
The input file is read with identifiers as text. `CustomerID` is the normalized name of the original `Customer ID`
(documented in Activity 2) and is stored as a nullable integer, so nulls stay null and are never turned into a fictitious
customer.

In [2]:
SRC = '../A-3-4_DataCleaning/outputs/transacciones_procesadas.csv'
tx = pd.read_csv(SRC, dtype={'Invoice': str, 'StockCode': str, 'Description': str, 'Country': str, 'MovementType': str})
tx['InvoiceDate'] = pd.to_datetime(tx['InvoiceDate'], format='%Y-%m-%d %H:%M:%S')
tx['Date'] = pd.to_datetime(tx['Date'], format='%Y-%m-%d')
tx['CustomerID'] = tx['CustomerID'].astype('Int64')
ROUTE = 'B'

print('File:', SRC, '| modified:', pd.Timestamp(os.path.getmtime(SRC), unit='s').strftime('%Y-%m-%d %H:%M'), '| route:', ROUTE)
print('Shape:', tx.shape)
print('Columns:', list(tx.columns))

# Integrity checks of the Activity 2 output
assert set(tx['MovementType']) == {'Purchase', 'Credit'}
assert tx['SourceRow'].is_unique
assert (tx['Date'] == tx['InvoiceDate'].dt.normalize()).all()          # original dates preserved
pur_all = tx[tx.MovementType == 'Purchase']; cre_all = tx[tx.MovementType == 'Credit']
assert (pur_all.Quantity > 0).all() and (pur_all.Price > 0).all() and (~pur_all.Invoice.str.startswith('C')).all()
assert (cre_all.Quantity < 0).all() and (cre_all.Price > 0).all() and cre_all.Invoice.str.startswith('C').all()
assert np.allclose(tx['LineAmount'], tx['Quantity'] * tx['Price'])
print('Period covered:', tx.InvoiceDate.min(), '→', tx.InvoiceDate.max())
display(tx.groupby('MovementType').agg(Lines=('SourceRow', 'size'), Invoices=('Invoice', 'nunique'), Amount=('LineAmount', 'sum')).round(2))
tx.head()

File: ../A-3-4_DataCleaning/outputs/transacciones_procesadas.csv | modified: 2026-10-31 01:34 | route: B
Shape: (1027016, 12)
Columns: ['SourceRow', 'Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'CustomerID', 'Country', 'LineAmount', 'Date', 'MovementType']


Period covered: 2009-12-01 07:45:00 → 2011-12-09 12:50:00


,Lines,Invoices,Amount
MovementType,,,
Credit,19103,8291,-1462424.18
Purchase,1007913,40077,20476260.45


,SourceRow,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,CustomerID,Country,LineAmount,Date,MovementType
0,1,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085,United Kingdom,83.4,2009-12-01,Purchase
1,2,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,81.0,2009-12-01,Purchase
2,3,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,81.0,2009-12-01,Purchase
3,4,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085,United Kingdom,100.8,2009-12-01,Purchase
4,5,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085,United Kingdom,30.0,2009-12-01,Purchase


In [3]:
# Lines and amount excluded from customer-level analysis for lack of identifier (kept in the Activity 2 global files)
nocust = tx[tx.CustomerID.isna()]
nc = nocust.groupby('MovementType').agg(Lines=('SourceRow', 'size'), Amount=('LineAmount', 'sum'))
nc.loc['Total'] = [len(nocust), nocust.LineAmount.sum()]
nc['% of lines'] = 100 * nc['Lines'] / len(tx)
display(nc.round(2))
cust = tx[tx.CustomerID.notna()].copy()
purchases = cust[cust.MovementType == 'Purchase']
credits = cust[cust.MovementType == 'Credit']
print(f'Lines with customer: {len(cust):,} | distinct customers: {cust.CustomerID.nunique():,} | with ≥1 valid purchase: {purchases.CustomerID.nunique():,}')
R['origen'] = dict(file='transacciones_procesadas.csv', route=ROUTE, rows=len(tx), purchases=len(pur_all), credits=len(cre_all),
                   purchase_inv=int(pur_all.Invoice.nunique()), credit_inv=int(cre_all.Invoice.nunique()),
                   period=[str(tx.InvoiceDate.min()), str(tx.InvoiceDate.max())],
                   nocust_lines=int(len(nocust)), nocust_amount=float(nocust.LineAmount.sum()),
                   nocust_pur_lines=int(nc.loc['Purchase', 'Lines']), nocust_pur_amount=float(nc.loc['Purchase', 'Amount']),
                   nocust_cre_lines=int(nc.loc['Credit', 'Lines']), nocust_cre_amount=float(nc.loc['Credit', 'Amount']),
                   nocust_pct=float(100 * len(nocust) / len(tx)), customers_any=int(cust.CustomerID.nunique()),
                   customers_pur=int(purchases.CustomerID.nunique()))

,Lines,Amount,% of lines
MovementType,,,
Credit,713.0,-377611.20,0.07
Purchase,228488.0,3101456.18,22.25
Total,229201.0,2723844.98,22.32


Lines with customer: 797,815 | distinct customers: 5,939 | with ≥1 valid purchase: 5,878


**Monetary definition (Route B):** `Monetary` and `TotalSpent` = signed sum of `LineAmount` of purchases **plus** credits
of the included customers within the corresponding period, i.e. **sales net of recorded credits**. Recency and Frequency
use valid purchases only; `Credit` movements never count as purchases. Periods are filtered first and aggregated after;
no reconciliation link is used to rewrite earlier purchases. Customer totals are rounded to the penny only to remove floating-point residue (e.g., 1e-13 instead of 0); prices and line amounts are never rounded.

---
## Section 1 – RFM dataset (Construcción de RFM para clustering)
### 1.1 Build
Full processed period up to 2011-12-09 inclusive. Reference date: **2011-12-10 00:00**. Days are computed on dates
normalized to midnight. Population: identified customers with at least one valid purchase in the period.

In [4]:
REF = pd.Timestamp('2011-12-10 00:00')
per = cust[cust.InvoiceDate < REF]                          # whole processed period
per_p = per[per.MovementType == 'Purchase']

# Orders: first collapse lines to customer-invoice, then to customer
orders = per_p.groupby(['CustomerID', 'Invoice'], as_index=False).agg(OrderDate=('Date', 'max'), Lines=('SourceRow', 'size'))
rfm = orders.groupby('CustomerID').agg(LastPurchase=('OrderDate', 'max'), FirstPurchase=('OrderDate', 'min'), Frequency=('Invoice', 'nunique'), Lines=('Lines', 'sum'))
rfm['Recency'] = (REF.normalize() - rfm['LastPurchase']).dt.days
included = rfm.index
elig = per[per.CustomerID.isin(included)]                   # purchases + credits of included customers
rfm['Monetary'] = elig.groupby('CustomerID')['LineAmount'].sum().round(2)   # cents: removes floating-point residue such as 1e-13
rfm['PurchaseAmount'] = elig[elig.MovementType == 'Purchase'].groupby('CustomerID')['LineAmount'].sum()
rfm['CreditAmount'] = elig[elig.MovementType == 'Credit'].groupby('CustomerID')['LineAmount'].sum().reindex(rfm.index).fillna(0)

clientes_rfm = rfm.reset_index()[['CustomerID', 'Recency', 'Frequency', 'Monetary']]
clientes_rfm.to_csv('outputs/clientes_rfm.csv', index=False)
print('Exported outputs/clientes_rfm.csv', clientes_rfm.shape)
clientes_rfm.head()

Exported outputs/clientes_rfm.csv (5878, 4)


,CustomerID,Recency,Frequency,Monetary
0,12346,326,12,-51.74
1,12347,3,8,4921.53
2,12348,76,5,2019.40
3,12349,19,4,4404.54
4,12350,311,1,334.40


### 1.2 RFM validation control

In [5]:
chk = pd.read_csv('outputs/clientes_rfm.csv')
excluded_credit_only = credits[~credits.CustomerID.isin(included)]
v = {
    'customers': int(rfm.shape[0]), 'file_rows': int(chk.shape[0]), 'file_cols': list(chk.columns),
    'null_ids': int(chk.CustomerID.isna().sum()), 'dup_ids': int(chk.CustomerID.duplicated().sum()),
    'ref': str(REF), 'first_purchase': str(per_p.InvoiceDate.min()), 'last_purchase': str(per_p.InvoiceDate.max()),
    'rec_min': int(chk.Recency.min()), 'rec_max': int(chk.Recency.max()), 'freq_min': int(chk.Frequency.min()), 'freq_max': int(chk.Frequency.max()),
    'monetary_sum': float(chk.Monetary.sum()), 'eligible_sum': float(elig.LineAmount.sum()),
    'eligible_pur': float(elig[elig.MovementType == 'Purchase'].LineAmount.sum()), 'eligible_cre': float(elig[elig.MovementType == 'Credit'].LineAmount.sum()),
    'credit_only_customers': int(excluded_credit_only.CustomerID.nunique()), 'credit_only_lines': int(len(excluded_credit_only)),
    'credit_only_amount': float(excluded_credit_only.LineAmount.sum()),
    'monetary_le0': int((chk.Monetary <= 0).sum()), 'monetary_lt0': int((chk.Monetary < 0).sum()), 'monetary_eq0': int((chk.Monetary == 0).sum()),
}
assert v['customers'] == v['file_rows'] and v['null_ids'] == 0 and v['dup_ids'] == 0
assert np.isclose(v['monetary_sum'], v['eligible_sum'])
assert v['freq_min'] >= 1 and v['rec_min'] >= 1
display(pd.Series(v, name='RFM checks').to_frame())
display(chk.describe().round(2))

,RFM checks
customers,5878
file_rows,5878
file_cols,"[CustomerID, Recency, Frequency, Monetary]"
null_ids,0
dup_ids,0
ref,2011-12-10 00:00:00
first_purchase,2009-12-01 07:45:00
last_purchase,2011-12-09 12:50:00
rec_min,1
rec_max,739


,CustomerID,Recency,Frequency,Monetary
count,5878.00,5878.00,5878.00,5878.00
mean,15315.31,201.87,6.29,2781.74
std,1715.57,209.35,13.01,13741.22
min,12346.00,1.00,1.00,-10953.50
25%,13833.25,26.00,1.00,329.68
50%,15314.50,96.00,3.00,838.86
75%,16797.75,380.00,7.00,2165.27
max,18287.00,739.00,398.00,570380.61


### 1.3 Customers with Monetary ≤ 0
They are **kept** for this review and flagged before later modelling. Their purchases and credits are shown, together with
the reconciliation status assigned to those credits in Activity 2 (`auditoria_conciliacion_abonos.csv`).

In [6]:
le0 = rfm[rfm.Monetary <= 0].copy()
audit = pd.read_csv('../A-3-4_DataCleaning/outputs/auditoria_conciliacion_abonos.csv', usecols=['SourceRow', 'Status', 'Reason'])
cr_le0 = credits[credits.CustomerID.isin(le0.index)].merge(audit, on='SourceRow', how='left')
cr_le0['ReturnedBeforeFirstPurchase'] = cr_le0.InvoiceDate < cr_le0.CustomerID.map(rfm['FirstPurchase'])
why = cr_le0.groupby('CustomerID').agg(CreditLines=('SourceRow', 'size'),
                                        UnreconciledAmount=('LineAmount', lambda s: s[cr_le0.loc[s.index, 'Status'] == 'Unreconciled'].sum()),
                                        NoEarlierPurchaseLines=('Reason', lambda s: (s == 'No earlier purchase of the product by the customer').sum()))
le0 = le0.join(why)
le0['Case'] = np.where(le0.Monetary == 0, 'Credits equal purchases (net zero)', 'Credits exceed purchases in the file')
display(le0[['Frequency', 'Recency', 'PurchaseAmount', 'CreditAmount', 'Monetary', 'CreditLines', 'UnreconciledAmount', 'NoEarlierPurchaseLines', 'Case']].round(2).sort_values('Monetary'))
print(le0['Case'].value_counts())
print('Credit lines of these customers by reconciliation status:'); display(cr_le0.groupby(['Status', 'Reason']).size().rename('Lines').to_frame())
R['rfm'] = v
R['rfm']['le0_cases'] = le0['Case'].value_counts().to_dict()
R['rfm']['le0_min'] = float(le0.Monetary.min())
R['rfm']['le0_min_id'] = int(le0.Monetary.idxmin())
R['rfm']['le0_status'] = cr_le0.groupby('Status').size().to_dict()
R['rfm']['le0_status_amount'] = cr_le0.groupby('Status')['LineAmount'].sum().to_dict()
R['rfm']['le0_credit_lines'] = int(len(cr_le0))
R['rfm']['le0_noearlier_lines'] = int((cr_le0.Reason == 'No earlier purchase of the product by the customer').sum())
R['rfm']['le0_eq0'] = int((le0.Monetary == 0).sum())
worst = le0.Monetary.idxmin()
print(f'\nMost negative case, customer {worst}:')
display(cust[cust.CustomerID == worst][['SourceRow', 'Invoice', 'StockCode', 'Description', 'Quantity', 'Price', 'LineAmount', 'InvoiceDate', 'MovementType']].merge(audit, on='SourceRow', how='left'))

,Frequency,Recency,PurchaseAmount,CreditAmount,Monetary,CreditLines,UnreconciledAmount,NoEarlierPurchaseLines,Case
CustomerID,,,,,,,,,
12918,1,627,10953.50,-21907.00,-10953.50,2,-10953.50,1,Credits exceed purchases in the file
15760,2,631,13916.34,-19712.21,-5795.87,3,-12754.04,1,Credits exceed purchases in the file
14063,9,439,22710.20,-26477.40,-3767.20,4,-13238.70,2,Credits exceed purchases in the file
15202,3,421,6001.49,-8571.67,-2570.18,5,-6346.50,2,Credits exceed purchases in the file
15413,5,692,6798.72,-9330.10,-2531.38,2,-9330.10,2,Credits exceed purchases in the file
14802,2,439,1502.98,-3005.96,-1502.98,4,-1502.98,2,Credits exceed purchases in the file
14255,1,547,1000.63,-2442.23,-1441.60,3,-2442.23,2,Credits exceed purchases in the file
14308,2,550,1147.02,-1996.47,-849.45,3,-1147.02,1,Credits exceed purchases in the file
15802,1,143,451.42,-902.84,-451.42,2,-451.42,1,Credits exceed purchases in the file


Case
Credits equal purchases (net zero)      21
Credits exceed purchases in the file    20
Name: count, dtype: int64
Credit lines of these customers by reconciliation status:


Lines
Status       Reason                                                   
Ambiguous    Several candidate invoices                              6
Reconciled   Single earlier invoice with sufficient balance        242
Unreconciled Earlier purchases only at a different price             8
             Insufficient earlier balance                            1
             No earlier purchase of the product by the customer    137


Most negative case, customer 12918:


,SourceRow,Invoice,StockCode,Description,Quantity,Price,LineAmount,InvoiceDate,MovementType,Status,Reason
0,135013,C502262,M,Manual,-1,10953.5,-10953.5,2010-03-23 15:20:00,Credit,Unreconciled,No earlier purchase of the product by the cust...
1,135014,502263,M,Manual,1,10953.5,10953.5,2010-03-23 15:22:00,Purchase,NaN,NaN
2,135015,C502264,M,Manual,-1,10953.5,-10953.5,2010-03-23 15:24:00,Credit,Reconciled,Single earlier invoice with sufficient balance


### 1.4 Manual check of two customers with different order counts (Question 1)
Customer **L** has few orders; customer **H** is frequent and its Monetary is markedly reduced by credits.

In [7]:
cands_low = rfm[(rfm.Frequency == 2) & (rfm.Lines > rfm.Frequency) & (rfm.CreditAmount == 0)]
cust_L = int(cands_low.sort_values('Lines', ascending=False).index[0])
rfm['CreditShare'] = -rfm.CreditAmount / rfm.PurchaseAmount
cands_high = rfm[(rfm.Frequency >= 20) & (rfm.Monetary > 0)]
cust_H = int(cands_high.sort_values('CreditShare', ascending=False).index[0])

manual = {}
for label, c in [('L', cust_L), ('H', cust_H)]:
    rows = per[per.CustomerID == c]
    inv = (rows.groupby(['Invoice', 'MovementType']).agg(Date=('InvoiceDate', 'max'), Lines=('SourceRow', 'size'), Units=('Quantity', 'sum'), Amount=('LineAmount', 'sum'))
               .reset_index().sort_values('Date'))
    p_inv = inv[inv.MovementType == 'Purchase']
    hand = dict(customer=c, valid_invoices=int(p_inv.Invoice.nunique()), purchase_lines=int(p_inv.Lines.sum()), credit_docs=int((inv.MovementType == 'Credit').sum()),
                credit_lines=int(inv[inv.MovementType == 'Credit'].Lines.sum()), last_purchase=str(p_inv.Date.max()),
                recency=int((REF.normalize() - p_inv.Date.max().normalize()).days),
                purchases=float(p_inv.Amount.sum()), credits=float(inv[inv.MovementType == 'Credit'].Amount.sum()))
    hand['monetary'] = hand['purchases'] + hand['credits']
    row = clientes_rfm.set_index('CustomerID').loc[c]
    assert hand['valid_invoices'] == row.Frequency and hand['recency'] == row.Recency and np.isclose(hand['monetary'], row.Monetary)
    manual[label] = hand
    print(f"\nCustomer {label} = {c}: {hand['valid_invoices']} valid invoices ({hand['purchase_lines']} purchase lines), last purchase {hand['last_purchase']}, "
          f"Recency {hand['recency']}, purchases £{hand['purchases']:,.2f} + credits £{hand['credits']:,.2f} = Monetary £{hand['monetary']:,.2f}")
    display(inv.head(12) if label == 'H' else inv)
    if label == 'H':
        print(f'... {len(inv)} documents in total; largest credits:')
        display(rows[rows.MovementType == 'Credit'].nsmallest(5, 'LineAmount')[['Invoice', 'StockCode', 'Description', 'Quantity', 'Price', 'LineAmount', 'InvoiceDate']])
print('\nFrequency if lines were counted instead of invoices:', {k: v['purchase_lines'] for k, v in manual.items()}, 'vs', {k: v['valid_invoices'] for k, v in manual.items()})
print(f"Across all customers: mean lines per order = {rfm.Lines.sum() / rfm.Frequency.sum():.1f}; median Frequency {rfm.Frequency.median():.0f} vs median line count {rfm.Lines.median():.0f}")
R['manual'] = manual
R['manual_global'] = dict(lines_per_order=float(rfm.Lines.sum() / rfm.Frequency.sum()), med_freq=float(rfm.Frequency.median()), med_lines=float(rfm.Lines.median()),
                          H_credit_share=float(rfm.loc[cust_H, 'CreditShare']), H_rank_freq=int((rfm.Frequency > rfm.loc[cust_H, 'Frequency']).sum() + 1))


Customer L = 12378: 2 valid invoices (301 purchase lines), last purchase 2011-08-02 10:34:00, Recency 130, purchases £5,416.32 + credits £0.00 = Monetary £5,416.32


,Invoice,MovementType,Date,Lines,Units,Amount
0,509980,Purchase,2010-05-26 14:07:00,82,787,1407.70
1,562046,Purchase,2011-08-02 10:34:00,219,2529,4008.62



Customer H = 17017: 29 valid invoices (378 purchase lines), last purchase 2011-12-07 09:32:00, Recency 3, purchases £12,576.99 + credits £-6,023.20 = Monetary £6,553.79


,Invoice,MovementType,Date,Lines,Units,Amount
0,490054,Purchase,2009-12-03 13:22:00,7,186,277.50
29,C491595,Credit,2009-12-11 13:14:00,1,-6,-29.70
30,C504637,Credit,2010-04-15 14:07:00,1,-1,-5843.70
1,512462,Purchase,2010-06-16 10:29:00,9,258,409.82
31,C512793,Credit,2010-06-18 09:39:00,2,-15,-18.58
2,523307,Purchase,2010-09-21 12:56:00,42,1002,1001.47
3,528177,Purchase,2010-10-21 10:46:00,14,247,278.57
4,528178,Purchase,2010-10-21 10:47:00,4,54,323.70
5,531831,Purchase,2010-11-09 18:29:00,15,252,498.00
6,531832,Purchase,2010-11-09 18:29:00,2,24,50.40


... 41 documents in total; largest credits:


,Invoice,StockCode,Description,Quantity,Price,LineAmount,InvoiceDate
156682,C504637,M,Manual,-1,5843.70,-5843.7,2010-04-15 14:07:00
25119,C491595,22138,BAKING SET 9 PIECE RETROSPOT,-6,4.95,-29.7,2009-12-11 13:14:00
691814,C554340,20979,36 PENCILS TUBE RED RETROSPOT,-16,1.25,-20.0,2011-05-23 17:40:00
739584,C559012,22900,SET 2 TEA TOWELS I LOVE LONDON,-6,3.25,-19.5,2011-07-05 12:50:00
521800,C538802,22960,JAM MAKING SET WITH JARS,-4,4.25,-17.0,2010-12-14 11:54:00



Frequency if lines were counted instead of invoices: {'L': 301, 'H': 378} vs {'L': 2, 'H': 29}
Across all customers: mean lines per order = 21.1; median Frequency 3 vs median line count 52


---
## Section 2 – Repurchase classification dataset (Clasificación de recompra y límites temporales)
### 2.1 Windows
| Use | Start (included) | End (excluded) |
|---|---|---|
| 180-day history | 2011-01-02 00:00 | 2011-07-01 00:00 |
| 90-day repurchase observation | 2011-07-01 00:00 | 2011-09-29 00:00 |

Rule: `start ≤ InvoiceDate < end` in both windows; they share no transactions.

In [8]:
H0, CUT, E1 = pd.Timestamp('2011-01-02'), pd.Timestamp('2011-07-01'), pd.Timestamp('2011-09-29')
assert (CUT - H0).days == 180 and (E1 - CUT).days == 90
# The input file covers both windows completely (no prior truncation)
print('File coverage:', tx.InvoiceDate.min(), '→', tx.InvoiceDate.max(), '| windows need', H0, '→', E1)
assert tx.InvoiceDate.min() < H0 and tx.InvoiceDate.max() >= E1
hist = cust[(cust.InvoiceDate >= H0) & (cust.InvoiceDate < CUT)]
obs = cust[(cust.InvoiceDate >= CUT) & (cust.InvoiceDate < E1)]
assert len(set(hist.SourceRow) & set(obs.SourceRow)) == 0
print('History rows:', len(hist), '| observation rows:', len(obs))
print('Days with data – history:', hist.Date.nunique(), '| observation:', obs.Date.nunique())

File coverage: 2009-12-01 07:45:00 → 2011-12-09 12:50:00 | windows need 2011-01-02 00:00:00 → 2011-09-29 00:00:00
History rows: 148539 | observation rows: 91380
Days with data – history: 147 | observation: 76


### 2.2 Population and predictors (history window only)

In [9]:
hist_p = hist[hist.MovementType == 'Purchase']
pop_orders = hist_p.groupby(['CustomerID', 'Invoice'], as_index=False).agg(OrderDate=('Date', 'max'))
pop = pop_orders.groupby('CustomerID').agg(LastHistPurchase=('OrderDate', 'max'), TotalOrders=('Invoice', 'nunique'))
pop['RecencyDays'] = (CUT.normalize() - pop['LastHistPurchase']).dt.days
hist_elig = hist[hist.CustomerID.isin(pop.index)]
pop['TotalSpent'] = hist_elig.groupby('CustomerID')['LineAmount'].sum().round(2)   # cents: removes floating-point residue
pop['HistPurchases'] = hist_elig[hist_elig.MovementType == 'Purchase'].groupby('CustomerID')['LineAmount'].sum()
pop['HistCredits'] = hist_elig[hist_elig.MovementType == 'Credit'].groupby('CustomerID')['LineAmount'].sum().reindex(pop.index).fillna(0)
print('Historical population:', len(pop), 'customers')
display(pop.describe().round(2))

Historical population: 2724 customers


,LastHistPurchase,TotalOrders,RecencyDays,TotalSpent,HistPurchases,HistCredits
count,2724,2724.00,2724.00,2724.00,2724.00,2724.00
mean,2011-05-01 05:37:47.841409,2.72,60.77,1163.98,1253.32,-89.34
min,2011-01-04 00:00:00,1.00,1.00,-1591.20,2.90,-77183.60
25%,2011-03-28 00:00:00,1.00,21.00,236.93,244.42,-9.90
50%,2011-05-15 00:00:00,2.00,47.00,482.02,492.36,0.00
75%,2011-06-10 00:00:00,3.00,95.00,1039.29,1067.41,0.00
max,2011-06-30 00:00:00,67.00,178.00,127365.23,127410.23,0.00
std,NaN,3.88,47.91,4125.93,4579.56,1753.58


### 2.3 Repurchase90Days label
Buyers after the cutoff = distinct `CustomerID` with at least one **valid purchase** (positive, no `C`) in the 90-day window.
A customer with only credits in that window does not qualify. The label is assigned with `isin()` over the full
historical table, so no customer is dropped and no row is multiplied.

In [10]:
obs_p = obs[obs.MovementType == 'Purchase']
buyers_after = pd.Index(obs_p.CustomerID.unique())
assert buyers_after.is_unique
pop['Repurchase90Days'] = pop.index.isin(buyers_after).astype('int64')

clientes_clasificacion = pop.reset_index()[['CustomerID', 'RecencyDays', 'TotalOrders', 'TotalSpent', 'Repurchase90Days']]
clientes_clasificacion.to_csv('outputs/clientes_clasificacion.csv', index=False)
print('Exported outputs/clientes_clasificacion.csv', clientes_clasificacion.shape)
clientes_clasificacion.head()

Exported outputs/clientes_clasificacion.csv (2724, 5)


,CustomerID,RecencyDays,TotalOrders,TotalSpent,Repurchase90Days
0,12346,164,1,0.00,0
1,12347,22,3,1494.16,1
2,12348,87,2,594.44,1
3,12350,149,1,334.40,0
4,12352,101,5,601.18,1


### 2.4 Classification validation control

In [11]:
cc = pd.read_csv('outputs/clientes_clasificacion.csv')
hist_rows_pop = hist_elig
w = {
    'rows': int(len(cc)), 'pop_distinct': int(hist_p.CustomerID.nunique()), 'cols': list(cc.columns),
    'null_ids': int(cc.CustomerID.isna().sum()), 'dup_ids': int(cc.CustomerID.duplicated().sum()),
    'labels': sorted(cc.Repurchase90Days.unique().tolist()), 'null_labels': int(cc.Repurchase90Days.isna().sum()),
    'rec_min': int(cc.RecencyDays.min()), 'rec_max': int(cc.RecencyDays.max()),
    'orders_min': int(cc.TotalOrders.min()), 'orders_max': int(cc.TotalOrders.max()), 'orders_int': str(cc.TotalOrders.dtype),
    'pred_min': str(hist_rows_pop.InvoiceDate.min()), 'pred_max': str(hist_rows_pop.InvoiceDate.max()),
    'pred_purchase_min': str(hist_p.InvoiceDate.min()), 'pred_purchase_max': str(hist_p.InvoiceDate.max()),
    'obs_min': str(obs_p.InvoiceDate.min()), 'obs_max': str(obs_p.InvoiceDate.max()),
    'after_cut_in_spent': int((hist_rows_pop.InvoiceDate >= CUT).sum()),
    'spent_sum': float(cc.TotalSpent.sum()), 'spent_recalc': float(hist_rows_pop.LineAmount.sum()),
    'spent_pur': float(hist_rows_pop[hist_rows_pop.MovementType == 'Purchase'].LineAmount.sum()),
    'spent_cre': float(hist_rows_pop[hist_rows_pop.MovementType == 'Credit'].LineAmount.sum()),
    'spent_na': int(cc.TotalSpent.isna().sum()),
    'spent_le0': int((cc.TotalSpent <= 0).sum()),
}
assert w['rows'] == w['pop_distinct'] and w['null_ids'] == 0 and w['dup_ids'] == 0
assert w['labels'] == [0, 1] and w['null_labels'] == 0
assert 1 <= w['rec_min'] and w['rec_max'] <= 180 and w['orders_min'] >= 1 and cc.TotalOrders.dtype.kind == 'i'
assert w['after_cut_in_spent'] == 0 and w['spent_na'] == 0
assert np.isclose(w['spent_sum'], w['spent_recalc'])
# Independent recomputation of TotalSpent customer by customer
indep = hist_rows_pop.groupby('CustomerID').LineAmount.sum().reindex(cc.CustomerID).values
assert np.allclose(indep, cc.TotalSpent.values, atol=0.005)
display(pd.Series(w, name='Classification checks').to_frame())

dist = cc.Repurchase90Days.value_counts().sort_index().rename('Customers').to_frame()
dist['Percent'] = (100 * dist['Customers'] / len(cc)).round(2)
dist.loc['Total'] = [len(cc), 100.0]
display(dist)
R['clas'] = w
R['clas']['dist'] = {str(k): dict(n=int(r.Customers), pct=float(100 * r.Customers / len(cc))) for k, r in dist.drop('Total').iterrows()}

,Classification checks
rows,2724
pop_distinct,2724
cols,"[CustomerID, RecencyDays, TotalOrders, TotalSp..."
null_ids,0
dup_ids,0
labels,"[0, 1]"
null_labels,0
rec_min,1
rec_max,178
orders_min,1


,Customers,Percent
Repurchase90Days,,
0,1315.0,48.27
1,1409.0,51.73
Total,2724.0,100.00


#### Customers with TotalSpent ≤ 0 under B
Reviewed with their historical purchases and credits; the value is kept as is (not made positive, no future returns used).

In [12]:
le0c = pop[pop.TotalSpent <= 0]
display(le0c[['TotalOrders', 'RecencyDays', 'HistPurchases', 'HistCredits', 'TotalSpent', 'Repurchase90Days']].round(2).sort_values('TotalSpent'))
cr_h = hist_elig[(hist_elig.MovementType == 'Credit') & hist_elig.CustomerID.isin(le0c.index)].merge(audit, on='SourceRow', how='left')
cr_h['BeforeFirstHistPurchase'] = cr_h.InvoiceDate < cr_h.CustomerID.map(pop_orders.groupby('CustomerID').OrderDate.min())
print('Their historical credits by Activity 2 reconciliation status:')
display(cr_h.groupby(['Status']).agg(Lines=('SourceRow', 'size'), Amount=('LineAmount', 'sum')).round(2))
R['clas']['le0_detail'] = dict(n=int(len(le0c)), eq0=int((le0c.TotalSpent == 0).sum()), min=float(le0c.TotalSpent.min()),
                               credit_lines=int(len(cr_h)), linked_outside=int((cr_h.Status != 'Reconciled').sum()),
                               status=cr_h.groupby('Status').size().to_dict())

,TotalOrders,RecencyDays,HistPurchases,HistCredits,TotalSpent,Repurchase90Days
CustomerID,,,,,,
12755,1,119,17.55,-1608.75,-1591.20,0
14146,1,93,662.30,-1241.43,-579.13,1
15197,1,91,216.75,-419.40,-202.65,0
13672,2,140,186.39,-257.85,-71.46,0
13762,1,57,306.72,-306.72,0.00,0
12346,1,164,77183.60,-77183.60,0.00,0
16163,1,71,415.00,-415.00,0.00,0
17190,1,46,195.00,-195.00,0.00,0
17548,1,72,103.30,-103.30,0.00,0


Their historical credits by Activity 2 reconciliation status:


,Lines,Amount
Status,,
Ambiguous,14,-587.35
Reconciled,44,-80992.56
Unreconciled,5,-151.14


### 2.5 One customer per class (Question 2)

In [13]:
obs_c = obs[obs.MovementType == 'Credit']
c1 = int(cc[(cc.Repurchase90Days == 1)].sort_values('CustomerID').CustomerID.iloc[0])
only_credit_after = sorted(set(obs_c.CustomerID.unique()) - set(buyers_after) & set(pop.index))
c0 = int(only_credit_after[0]) if only_credit_after else int(cc[cc.Repurchase90Days == 0].CustomerID.iloc[0])
show = ['Invoice', 'StockCode', 'Quantity', 'Price', 'LineAmount', 'InvoiceDate', 'MovementType']
ex = {}
for lbl, c in [('1', c1), ('0', c0)]:
    h = hist[hist.CustomerID == c].groupby(['Invoice', 'MovementType']).agg(Date=('InvoiceDate', 'max'), Lines=('SourceRow', 'size'), Amount=('LineAmount', 'sum')).reset_index().sort_values('Date')
    o = obs[obs.CustomerID == c].groupby(['Invoice', 'MovementType']).agg(Date=('InvoiceDate', 'max'), Lines=('SourceRow', 'size'), Amount=('LineAmount', 'sum')).reset_index().sort_values('Date')
    print(f"\n=== Class {lbl}: customer {c} | output row:"); display(cc[cc.CustomerID == c])
    print('History window documents:'); display(h)
    print('Observation window documents:'); display(o)
    ex[lbl] = dict(customer=c, hist_purchase_inv=int((h.MovementType == 'Purchase').sum()), hist_last=str(h[h.MovementType == 'Purchase'].Date.max()),
                   obs_purchase_inv=int((o.MovementType == 'Purchase').sum()), obs_credit_docs=int((o.MovementType == 'Credit').sum()),
                   obs_first_purchase=(str(o[o.MovementType == 'Purchase'].Date.min()) if (o.MovementType == 'Purchase').any() else None),
                   obs_credit_inv=o[o.MovementType == 'Credit'].Invoice.tolist(), obs_credit_amount=float(o[o.MovementType == 'Credit'].Amount.sum()),
                   row=cc[cc.CustomerID == c].iloc[0].to_dict())
assert ex['1']['obs_purchase_inv'] >= 1 and ex['0']['obs_purchase_inv'] == 0 and ex['0']['hist_purchase_inv'] >= 1
print('Class-0 customers whose only movement after the cutoff is a credit:', len(only_credit_after))
R['ejemplos'] = ex
R['ejemplos']['only_credit_after'] = len(only_credit_after)


=== Class 1: customer 12347 | output row:


,CustomerID,RecencyDays,TotalOrders,TotalSpent,Repurchase90Days
1,12347,22,3,1494.16,1


History window documents:


,Invoice,MovementType,Date,Lines,Amount
0,542237,Purchase,2011-01-26 14:30:00,29,475.39
1,549222,Purchase,2011-04-07 10:43:00,24,636.25
2,556201,Purchase,2011-06-09 13:01:00,18,382.52


Observation window documents:


,Invoice,MovementType,Date,Lines,Amount
0,562032,Purchase,2011-08-02 08:48:00,22,584.91


... (3 outputs left out)

,Invoice,MovementType,Date,Lines,Amount
0,556127,Purchase,2011-06-09 10:23:00,49,1621.25


Observation window documents:


,Invoice,MovementType,Date,Lines,Amount
0,C562728,Credit,2011-08-09 09:41:00,3,-134.55


Class-0 customers whose only movement after the cutoff is a credit: 28


### 2.6 Leakage checks (Question 3)
What would change if post-cutoff information leaked into the predictors, and whether new post-cutoff customers created rows.

In [14]:
# (a) Leaky TotalSpent including the observation window (e.g., August purchases)
leak_spent = cust[(cust.InvoiceDate >= H0) & (cust.InvoiceDate < E1) & cust.CustomerID.isin(pop.index)].groupby('CustomerID').LineAmount.sum().reindex(pop.index)
aug = cust[(cust.InvoiceDate >= '2011-08-01') & (cust.InvoiceDate < '2011-09-01') & (cust.MovementType == 'Purchase') & cust.CustomerID.isin(pop.index)]
# (b) Leaky Recency using the last purchase up to the end of the label window
leak_last = cust[(cust.MovementType == 'Purchase') & (cust.InvoiceDate < E1) & cust.CustomerID.isin(pop.index)].groupby('CustomerID').Date.max()
leak_rec = (CUT.normalize() - leak_last).dt.days
cmp_ = pd.DataFrame({'Correct TotalSpent': pop.TotalSpent, 'Leaky TotalSpent': leak_spent, 'Correct RecencyDays': pop.RecencyDays,
                     'Leaky RecencyDays': leak_rec, 'Repurchase90Days': pop.Repurchase90Days})
summary = cmp_.groupby('Repurchase90Days').agg(['mean', 'min', 'max']).round(1)
display(summary)
leak = dict(aug_customers=int(aug.CustomerID.nunique()), aug_amount=float(aug.LineAmount.sum()),
            spent_changed=int((~np.isclose(leak_spent, pop.TotalSpent)).sum()),
            leak_rec_nonpos_class1=int((leak_rec[pop.Repurchase90Days == 1] <= 0).sum()), class1=int((pop.Repurchase90Days == 1).sum()),
            leak_rec_nonpos_class0=int((leak_rec[pop.Repurchase90Days == 0] <= 0).sum()),
            mean_spent_c1_ok=float(pop.loc[pop.Repurchase90Days == 1, 'TotalSpent'].mean()), mean_spent_c1_leak=float(leak_spent[pop.Repurchase90Days == 1].mean()),
            mean_spent_c0_ok=float(pop.loc[pop.Repurchase90Days == 0, 'TotalSpent'].mean()), mean_spent_c0_leak=float(leak_spent[pop.Repurchase90Days == 0].mean()))
print(leak)
ex1 = ex['1']['customer']
print(f'Example class-1 customer {ex1}: correct TotalSpent £{pop.loc[ex1, "TotalSpent"]:,.2f} vs leaky £{leak_spent[ex1]:,.2f}; correct RecencyDays {pop.loc[ex1, "RecencyDays"]} vs leaky {leak_rec[ex1]}')
leak['ex1'] = dict(c=ex1, ok_spent=float(pop.loc[ex1, 'TotalSpent']), leak_spent=float(leak_spent[ex1]), ok_rec=int(pop.loc[ex1, 'RecencyDays']), leak_rec=int(leak_rec[ex1]))

# (c) New customers after the cutoff
first_ever = purchases.groupby('CustomerID').InvoiceDate.min()
new_after = first_ever[(first_ever >= CUT) & (first_ever < E1)].index
buyers_not_in_pop = buyers_after.difference(pop.index)
leak.update(new_after=int(len(new_after)), new_after_in_output=int(cc.CustomerID.isin(new_after).sum()),
            buyers_after=int(len(buyers_after)), buyers_not_in_pop=int(len(buyers_not_in_pop)), buyers_not_in_pop_in_output=int(cc.CustomerID.isin(buyers_not_in_pop).sum()),
            returning_not_in_pop=int(len(set(buyers_not_in_pop) - set(new_after))))
assert leak['new_after_in_output'] == 0 and leak['buyers_not_in_pop_in_output'] == 0 and len(cc) == len(pop)
print(f"Distinct buyers in the observation window: {leak['buyers_after']:,}; of them {leak['buyers_not_in_pop']:,} are not in the historical population "
      f"({leak['new_after']:,} first-time customers + {leak['returning_not_in_pop']:,} returning customers inactive in the history window); rows created for them: {leak['buyers_not_in_pop_in_output']}")
# Check that a left merge would give the same result (no lost or multiplied rows)
m = pop.reset_index()[['CustomerID']].merge(pd.DataFrame({'CustomerID': buyers_after, 'flag': 1}), on='CustomerID', how='left')
assert len(m) == len(pop) and (m.flag.fillna(0).astype(int).values == pop.Repurchase90Days.values).all()
inner = pop.reset_index()[['CustomerID']].merge(pd.DataFrame({'CustomerID': buyers_after}), on='CustomerID', how='inner')
print('An inner join would keep only', len(inner), 'rows and lose the whole class 0.')
leak['inner_rows'] = int(len(inner))
R['leak'] = leak

Correct TotalSpent                   Leaky TotalSpent                   Correct RecencyDays          Leaky RecencyDays         
                               mean     min       max             mean     min       max                mean min  max              mean min  max
Repurchase90Days                                                                                                                                
0                             501.3 -1591.2   21535.9            500.3 -1591.2   21535.9                75.1   1  178              75.1   1  178
1                            1782.5  -579.1  127365.2           2972.4    38.7  193797.8                47.4   1  178             -58.7 -89    0

{'aug_customers': 712, 'aug_amount': 544948.0800000001, 'spent_changed': 1436, 'leak_rec_nonpos_class1': 1409, 'class1': 1409, 'leak_rec_nonpos_class0': 0, 'mean_spent_c1_ok': 1782.4876721078779, 'mean_spent_c1_leak': 2972.4453804116397, 'mean_spent_c0_ok': 501.25135361216735, 'mean_spent_c0_leak': 500.33354448669206}
Example class-1 customer 12347: correct TotalSpent £1,494.16 vs leaky £2,079.07; correct RecencyDays 22 vs leaky -32
Distinct buyers in the observation window: 2,089; of them 680 are not in the historical population (373 first-time customers + 307 returning customers inactive in the history window); rows created for them: 0
An inner join would keep only 1409 rows and lose the whole class 0.


### 2.7 Context for interpreting class 0 (Question 4)
Validation-only check, **not** used in any predictor or label: how many class-0 customers buy again after the 90-day horizon
(2011-09-29 to 2011-12-09). It illustrates that 0 means "no purchase in this horizon", not definitive churn.

In [15]:
late = purchases[(purchases.InvoiceDate >= E1) & (purchases.InvoiceDate < REF)]
c0_ids = cc.loc[cc.Repurchase90Days == 0, 'CustomerID']
c0_late = int(c0_ids.isin(late.CustomerID.unique()).sum())
print(f'Class-0 customers with a valid purchase between 2011-09-29 and 2011-12-09: {c0_late:,} of {len(c0_ids):,} ({100 * c0_late / len(c0_ids):.1f}%)')
R['clase0_late'] = dict(n=c0_late, total=int(len(c0_ids)), pct=float(100 * c0_late / len(c0_ids)))

Class-0 customers with a valid purchase between 2011-09-29 and 2011-12-09: 462 of 1,315 (35.1%)


---
## Section 3 – Dataset comparison and export (Entrega)

In [16]:
print('clientes_rfm.csv – first five rows'); display(pd.read_csv('outputs/clientes_rfm.csv').head())
print('clientes_clasificacion.csv – first five rows'); display(pd.read_csv('outputs/clientes_clasificacion.csv').head())
rfm_ids, cls_ids = set(clientes_rfm.CustomerID), set(clientes_clasificacion.CustomerID)
comp = dict(rfm_rows=len(clientes_rfm), rfm_cols=clientes_rfm.shape[1], cls_rows=len(clientes_clasificacion), cls_cols=clientes_clasificacion.shape[1],
            cls_subset_of_rfm=bool(cls_ids <= rfm_ids), only_rfm=len(rfm_ids - cls_ids),
            only_rfm_before=int((rfm.loc[list(rfm_ids - cls_ids), 'LastPurchase'] < H0).sum()),
            only_rfm_first_after=int((rfm.loc[list(rfm_ids - cls_ids), 'FirstPurchase'] >= CUT).sum()))
comp['only_rfm_gap'] = comp['only_rfm'] - comp['only_rfm_before'] - comp['only_rfm_first_after']
print(comp)
# Scope lost for lack of customer identifier inside each period
nocust_hist = nocust[(nocust.InvoiceDate >= H0) & (nocust.InvoiceDate < CUT)]
comp.update(nocust_hist_lines=int(len(nocust_hist)), nocust_hist_amount=float(nocust_hist.LineAmount.sum()),
            hist_total_amount=float(tx[(tx.InvoiceDate >= H0) & (tx.InvoiceDate < CUT)].LineAmount.sum()),
            total_amount=float(tx.LineAmount.sum()))
R['comp'] = comp
with open('outputs/resultados_actividad3.json', 'w', encoding='utf-8') as fh:
    json.dump(R, fh, ensure_ascii=False, indent=1, default=str)
print('Saved outputs/resultados_actividad3.json')

clientes_rfm.csv – first five rows


,CustomerID,Recency,Frequency,Monetary
0,12346,326,12,-51.74
1,12347,3,8,4921.53
2,12348,76,5,2019.40
3,12349,19,4,4404.54
4,12350,311,1,334.40


clientes_clasificacion.csv – first five rows


,CustomerID,RecencyDays,TotalOrders,TotalSpent,Repurchase90Days
0,12346,164,1,0.00,0
1,12347,22,3,1494.16,1
2,12348,87,2,594.44,1
3,12350,149,1,334.40,0
4,12352,101,5,601.18,1


{'rfm_rows': 5878, 'rfm_cols': 4, 'cls_rows': 2724, 'cls_cols': 5, 'cls_subset_of_rfm': True, 'only_rfm': 3154, 'only_rfm_before': 1659, 'only_rfm_first_after': 837, 'only_rfm_gap': 658}
Saved outputs/resultados_actividad3.json


### Generated files
| File | Rows × columns | Key and period | Notebook section |
|---|---|---|---|
| `outputs/clientes_rfm.csv` | one row per customer × 4 (`CustomerID`, `Recency`, `Frequency`, `Monetary`) | `CustomerID`; 2009-12-01 to 2011-12-09, reference 2011-12-10 | Section 1 |
| `outputs/clientes_clasificacion.csv` | one row per customer × 5 (`CustomerID`, `RecencyDays`, `TotalOrders`, `TotalSpent`, `Repurchase90Days`) | `CustomerID`; history 2011-01-02 to 2011-07-01 (excl.), label 2011-07-01 to 2011-09-29 (excl.) | Section 2 |
| `outputs/resultados_actividad3.json` | — | Figures used in the report | Section 3 |

**Activity limit:** no train/test split, stratification, transformations, algorithms, metrics or temporal covariates for
regression are built here.